In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne as mne

from plotnine import (
    aes,
    element_text,
    facet_grid,
    geom_line,
    geom_ribbon,
    geom_vline,
    ggplot,
    labs,
    scale_x_log10,
    theme,
    theme_bw,
)

from pesco.io import load_ieeg, load_sources
from pesco.preprocess import get_psd_interval


def find_project_root(start: Path) -> Path:
    """Walk upward until we find the repo root."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root with pyproject.toml and data folder")


def lobes_long_compat(raw, region_df):
    """Compatibility helper for old utils.lobes_long behavior after refactor."""
    psd_df, _ = get_psd_interval(raw, save_psd=False)
    psd_df = psd_df.copy()
    psd_df["id"] = psd_df.index
    psd_long = psd_df.melt(id_vars=["id"], var_name="frequency", value_name="spectral_density")
    result_long = psd_long.set_index("id").join(region_df.set_index("ch_name"))
    result_long["frequency"] = pd.to_numeric(result_long["frequency"], errors="coerce")
    return result_long.reset_index(drop=True)


#%%
PROJECT_PATH = find_project_root(Path.cwd().resolve())
OUT_PATH = PROJECT_PATH / "data" / "preproc"
OUT_PATH.mkdir(parents=True, exist_ok=True)

raw1, lobes1 = load_ieeg(PROJECT_PATH / "data" / "Frauscher2018", OUT_PATH)
raw2, lobes2 = load_sources(PROJECT_PATH / "data" / "Mantini2018", OUT_PATH)

result_long1 = lobes_long_compat(raw1, lobes1)
result_long2 = lobes_long_compat(raw2, lobes2)
result = pd.concat([result_long1, result_long2], axis=0, ignore_index=True)


#%%
scalings = 'auto'  # Could also pass a dictionary with some value == 'auto'
x = raw1.plot(
    n_channels=100,
    scalings=scalings,
    title='Auto-scaled Data from arrays',
    show=True,
    block=False,
)


#%%
datatoplot = result.copy()
datatoplot['subindex'] = datatoplot.groupby(['Lobe', 'frequency']).cumcount()

summary = (
    datatoplot.groupby(['Lobe', 'dataset', 'frequency'], as_index=False)
    .agg(
        median=('spectral_density', lambda s: np.nanmedian(s.to_numpy())),
        q05=('spectral_density', lambda s: np.nanquantile(s.to_numpy(), 0.05)),
        q95=('spectral_density', lambda s: np.nanquantile(s.to_numpy(), 0.95)),
        minimum=('spectral_density', lambda s: np.nanmin(s.to_numpy())),
        maximum=('spectral_density', lambda s: np.nanmax(s.to_numpy())),
    )
)

g = (
    ggplot(summary, aes(x='frequency', y='median', color='dataset', fill='dataset'))
    + geom_ribbon(aes(ymin='q05', ymax='q95'), alpha=0.2, color=None)
    + geom_line(size=0.8)
    + geom_line(aes(y='minimum'), linetype='dashed', alpha=0.7)
    + geom_line(aes(y='maximum'), linetype='dashed', alpha=0.7)
    + geom_vline(xintercept=[4, 8, 13, 40], linetype='dashed', color='black', size=0.3)
    + facet_grid('Lobe ~ dataset', scales='free_y')
    + scale_x_log10(breaks=[4, 8, 13, 30, 80])
    + labs(x='Frequency (Hz)', y='Spectral density')
    + theme_bw()
    + theme(
        figure_size=(14, 10),
        axis_text_x=element_text(size=8),
        axis_text_y=element_text(size=8),
    )
)

g

Creating RawArray with float64 data, n_channels=1772, n_times=13600
    Range : 0 ... 13599 =      0.000 ...    67.995 secs
Ready.
Overwriting existing file.
Writing /Users/daniel/PhD/spectral-comparison/code/data/preproc/ieeg_raw.fif
Overwriting existing file.
Closing /Users/daniel/PhD/spectral-comparison/code/data/preproc/ieeg_raw.fif
[done]


Exception ignored while calling weakref callback <function WeakValueDictionary.__init__.<locals>.remove at 0x11ae0b690>:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.4/Frameworks/Python.framework/Versions/3.14/lib/python3.14/weakref.py", line 105, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 


Creating RawArray with float64 data, n_channels=1444, n_times=62689
    Range : 0 ... 62688 =      0.000 ...   313.440 secs
Ready.
Overwriting existing file.
Writing /Users/daniel/PhD/spectral-comparison/code/data/preproc/sources_raw.fif
Overwriting existing file.
Closing /Users/daniel/PhD/spectral-comparison/code/data/preproc/sources_raw.fif
[done]


KeyError: "None of ['ch_name'] are in the columns"